In [1]:
from imolcryff.calculator import DihedralCalculator, load_g16scan
from imolcryff.io.rdkit import atoms2rdkit
from imolcryff.trainer.dmff_utils import  *
from imolcryff.trainer import mse_energy, DihedralTrainer
from ase.io import read
from ase.visualize import view
from jax import value_and_grad, jit, vmap

import matplotlib.pyplot as plt

/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/psi4/driver/procrouting/solvent/ddx.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/Users/srak/apl/iMolCRAFT/DMFF/dmff/admp/qeq.py:33: UserWarning: jaxopt not found, QEQ cannot be used.
  warnings.warn("jaxopt not found, QEQ cannot be used.")
Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.


In [2]:
pdbfile = "dmc.pdb"
atoms = read(pdbfile)
mol, mol2d, _ = atoms2rdkit(atoms)
calculator = DihedralCalculator(atoms, mol, "dmc" "temp_directory")
calculator.scan_list

[[1, 0, 2, 8], [1, 0, 3, 4], [0, 2, 8, 9], [0, 3, 4, 5]]

In [3]:
# check dihedral atom index -> 1,0,2,8
view(load_g16scan("dmc_dihed_0.log")[2])
# check dihedral atom index -> 0,2,8,9
view(load_g16scan("dmc_dihed_2.log")[2])

<Popen: returncode: None args: ['/Users/srak/miniconda3/envs/imc_dev/bin/pyt...>

In [4]:
calculator.scan_list  = [calculator.scan_list[0] , calculator.scan_list[2]]
calculator.qm_calculators = [calculator.qm_calculators[0], calculator.qm_calculators[2]]
calculator.ff_calculators = [calculator.ff_calculators[0], calculator.ff_calculators[2]]
calculator.qm_scan = [calculator.qm_scan[0], calculator.qm_scan[2]]
calculator.ff_scan = [calculator.ff_scan[0], calculator.ff_scan[2]]

calculator.qm_scan[0]["angles_deg"], \
calculator.qm_scan[0]["energies_kjmol"], \
calculator.qm_scan[0]["atoms"], \
      = load_g16scan("dmc_dihed_0.log")

calculator.qm_scan[1]["angles_deg"], \
calculator.qm_scan[1]["energies_kjmol"], \
calculator.qm_scan[1]["atoms"], \
      = load_g16scan("dmc_dihed_2.log")

In [5]:
def loss_energy(ffparams, ff, topology, positions, pairs, y_gt):
    ff_d = update_ffinfo_from_params(ff, ffparams)
    ffparams_wo_charge = {}
    for key in ffparams.keys():
        if key == "NonbondedForce":
            ffparams_wo_charge[key] = {}
            for key2 in ffparams[key].keys():
                if key2 != "charges":
                    ffparams_wo_charge[key][key2] = ffparams[key][key2]
        elif key == "VsiteForce":
            pass
        else:
            ffparams_wo_charge[key] = ffparams[key]

    pots = ff_d.createPotential(topology) # should be pdb topology wo vsites
    efunc = pots.getPotentialFunc()
    batched_efunc = vmap(lambda x: efunc(x, None, pairs[0], ffparams_wo_charge))
    e_ff = batched_efunc(positions)
    e_ff = e_ff 
    y_gt = y_gt 
    print(f"e_ff: {e_ff}")
    print(f"y_gt: {y_gt}")
    loss = mse_energy(e_ff, y_gt, weight_scheme="uniform", norm_var=True)
    print(f"loss: {loss}")
    return loss

In [6]:
tr = DihedralTrainer(
    ffxml="dmc.xml",
    pdbfile=pdbfile,
    calculator=calculator,
    loss_fn=loss_energy,
    relax_steps=20,
)

In [7]:
tr.setup()

AssertionError: The length of angles and atoms_list must be the same.

In [7]:
tr.fit(steps=1000, relax_steps=20)

epoch 0
loss 52.3120971278295
epoch 20
loss 5.310958401009366
epoch 40
loss 2.1308386308138427
epoch 60
loss 1.020402552877622
epoch 80
loss 0.6193939472946822
epoch 100
loss 0.5000922775617563
epoch 120
loss 0.4519327957774672
epoch 140
loss 0.41191072785239347
epoch 160
loss 0.3861397106121494
epoch 180
loss 0.34217152703712883
epoch 200
loss 0.31437611465016374
epoch 220
loss 0.2895376683182976
epoch 240
loss 0.2670055238365105
epoch 260
loss 0.24278317233830576


KeyboardInterrupt: 

In [ ]:
plt.plot(tr.calculator.qm_dihedscan[0]["energies_kjmol"])
plt.plot(tr.calculator.ff_dihedscan[0]["energies_kjmol"])

In [ ]:
plt.plot(tr.calculator.qm_dihedscan[1]["energies_kjmol"])
plt.plot(tr.calculator.ff_dihedscan[1]["energies_kjmol"])